# Ai Data Quality Monitor

In [3]:
!pip install -q transformers datasets accelerate scikit-learn

In [4]:
# import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

In [5]:
# load dataset

from google.colab import files

uploaded = files.upload()

Saving online_retail_II.xlsx to online_retail_II.xlsx


In [10]:
import pandas as pd

file_path = "online_retail_II.xlsx"

excel_file = pd.ExcelFile(file_path)

print(excel_file.sheet_names)

['Year 2009-2010', 'Year 2010-2011']


In [11]:
df = pd.read_excel(file_path, sheet_name=0)

print(df.shape)
print(df.head())

(525461, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [12]:
# dataset shape

print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

Total Rows: 525461
Total Columns: 8


In [13]:
# check column names
print(df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [14]:
# check datatypes
df.dtypes

,0
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,object


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [16]:
# check missing values
df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,2928
Quantity,0
InvoiceDate,0
Price,0
Customer ID,107927
Country,0


In [17]:
# check duplicates
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 6865


In [18]:
# check outliers
for col in df.select_dtypes(include=np.number).columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((df[col] < lower) | (df[col] > upper)).sum()

    print(col, ":", count, "possible outliers")

Quantity : 57870 possible outliers
Price : 35273 possible outliers
Customer ID : 0 possible outliers


In [19]:
# check text and category issue
for col in df.select_dtypes(include="object").columns:
    values = df[col].dropna().astype(str)

    cleaned = values.str.strip().str.lower()

    if cleaned.nunique() < values.nunique():
        print("Possible text issues in:", col)

Possible text issues in: StockCode
Possible text issues in: Description


In [20]:
# select text column for Transformer
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Available text columns:", text_columns)

Available text columns: ['Invoice', 'StockCode', 'Description', 'Country']


In [21]:
text_column = "Description"

texts = df[text_column].dropna().astype(str).tolist()

print("Total text samples:", len(texts))

Total text samples: 522533


In [22]:
# create clean & noisy text Samples
def make_noisy_text(text):
    return text + " " + text

clean_texts = texts
noisy_texts = [make_noisy_text(text) for text in texts]

print("Clean samples:", len(clean_texts))
print("Noisy samples:", len(noisy_texts))

Clean samples: 522533
Noisy samples: 522533


In [23]:
# prepare Transformer training data
training_texts = clean_texts + noisy_texts

labels = (
    [0] * len(clean_texts) +
    [1] * len(noisy_texts)
)

print("Total training samples:", len(training_texts))

Total training samples: 1045066


In [24]:
# load pretrained DistiBERT
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("Transformer model loaded!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Transformer model loaded!


In [25]:
# split Training data
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    training_texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training samples:", len(train_texts))
print("Testing samples:", len(val_texts))

Training samples: 836052
Testing samples: 209014


In [26]:
# Tokenize text
train_data = tokenizer(
    train_texts,
    padding=True,
    truncation=True,
    max_length=64
)

val_data = tokenizer(
    val_texts,
    padding=True,
    truncation=True,
    max_length=64
)

print("Text tokenization completed!")

Text tokenization completed!


In [27]:
from datasets import Dataset

train_dict = dict(train_data)
train_dict["labels"] = list(train_labels)
train_dataset = Dataset.from_dict(train_dict)

val_dict = dict(val_data)
val_dict["labels"] = list(val_labels)
val_dataset = Dataset.from_dict(val_dict)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 836052
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 209014
})


In [28]:
# Use a smaller subset for faster testing
train_dataset = train_dataset.shuffle(seed=42).select(
    range(min(2000, len(train_dataset)))
)

val_dataset = val_dataset.shuffle(seed=42).select(
    range(min(500, len(val_dataset)))
)

In [29]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/training_results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    save_strategy="epoch",
    report_to="none"
)

print("Training arguments created successfully!")

Training arguments created successfully!


In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Trainer created successfully!")

Trainer created successfully!


In [31]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.035970


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.03597026443481445, metrics={'train_runtime': 417.2008, 'train_samples_per_second': 4.794, 'train_steps_per_second': 1.198, 'total_flos': 15523523280000.0, 'train_loss': 0.03597026443481445, 'epoch': 1.0})

In [32]:
model.save_pretrained("text_quality_model")
tokenizer.save_pretrained("text_quality_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('text_quality_model/tokenizer_config.json',
 'text_quality_model/tokenizer.json')

In [33]:
# Evaluate model
results = trainer.evaluate()

print(results)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
0.035970,0.039497,500


{'eval_loss': 0.039497409015893936}


In [34]:
import shutil

shutil.make_archive(
    "/content/text_quality_model",
    "zip",
    "/content/text_quality_model"
)

'/content/text_quality_model.zip'

In [35]:
from google.colab import files

files.download("/content/text_quality_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>